In [1]:
!pip install yt-dlp pydub tqdm


Defaulting to user installation because normal site-packages is not writeable


You should consider upgrading via the 'c:\program files\python38\python.exe -m pip install --upgrade pip' command.


In [3]:
import os
import yt_dlp
import math
from glob import glob
from pydub import AudioSegment
from pydub.utils import which

# ✅ FFmpeg Setup
FFMPEG_EXE_PATH = r"C:\Users\gaurav\Desktop\ffmpeg.exe"
os.environ["PATH"] += os.pathsep + os.path.dirname(FFMPEG_EXE_PATH)
AudioSegment.converter = FFMPEG_EXE_PATH
print("🔧 FFmpeg path detected by Pydub:", which("ffmpeg"))

# ✅ Refined Queries for Baby Choking / Uncomfortable Sounds
CLASSES = {
    "man-woman-ailing-screaming-in-pain": [
        "man screaming in pain sound effect",
        "woman screaming in pain audio",
        "man yelling in agony",
        "woman crying out in pain",
        "realistic pain scream man",
        "real human scream of pain",
        "injured man screaming sound",
        "woman ailing scream effect",
        "emergency trauma scream audio",
        "person screaming in suffering sound"
    ]
}





CLIPS_PER_CLASS = 2000
CLIP_DURATION_MS = 15 * 1000
BATCH_SIZE = 500
TEMP_DIR = "temp_audio_files"
os.makedirs(TEMP_DIR, exist_ok=True)

# ✅ Get batch folder path
def get_batch_folder(class_name, index):
    batch_num = index // BATCH_SIZE + 1
    return f"{class_name}_batch_{batch_num}"

# ✅ Download + Slice into 15s audio clips
def download_and_split_audio(class_name, search_query):
    total_clips = sum(len(glob(os.path.join(f"{class_name}_batch_{i}", "*.wav")))
                      for i in range(1, math.ceil(CLIPS_PER_CLASS / BATCH_SIZE) + 1))

    if total_clips >= CLIPS_PER_CLASS:
        print(f"✅ {class_name}: Already has {total_clips} clips.")
        return

    temp_wav_path = os.path.join(TEMP_DIR, f"{class_name}_temp.wav")

    ydl_opts = {
        'format': 'bestaudio/best',
        'quiet': True,
        'outtmpl': os.path.join(TEMP_DIR, f"{class_name}_temp.%(ext)s"),
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'wav',
            'preferredquality': '192',
        }],
        'ffmpeg_location': FFMPEG_EXE_PATH,
        'noplaylist': True,
        'default_search': 'ytsearch20',
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        print(f"🔍 [{class_name}] Searching YouTube for: {search_query}")
        info = ydl.extract_info(search_query, download=False)
        entries = info.get('entries', [])

        audio_file_counter = total_clips + 1

        for entry in entries:
            if total_clips >= CLIPS_PER_CLASS:
                break

            try:
                url = entry['webpage_url']
                print(f"⬇ [{class_name}] Downloading from: {entry['title']}")
                ydl.download([url])

                audio = AudioSegment.from_wav(temp_wav_path)

                for i in range(0, len(audio), CLIP_DURATION_MS):
                    if total_clips >= CLIPS_PER_CLASS:
                        break

                    chunk = audio[i:i + CLIP_DURATION_MS]
                    if len(chunk) < CLIP_DURATION_MS:
                        continue

                    batch_folder = get_batch_folder(class_name, total_clips)
                    os.makedirs(batch_folder, exist_ok=True)

                    clip_name = f"{class_name}_audio_file{audio_file_counter}.wav"
                    clip_path = os.path.join(batch_folder, clip_name)
                    chunk.export(clip_path, format="wav")
                    print(f"✅ [{class_name}] Saved: {clip_name} ({total_clips+1}/{CLIPS_PER_CLASS})")

                    total_clips += 1
                    audio_file_counter += 1

                if os.path.exists(temp_wav_path):
                    os.remove(temp_wav_path)

            except Exception as e:
                print(f"⚠ [{class_name}] Error processing video: {e}")

# ✅ Process all search queries
for class_name, queries in CLASSES.items():
    for query in queries:
        download_and_split_audio(class_name, query)

# ✅ Final summary
print("\n📊 Final Clip Counts:")
for class_name in CLASSES:
    batches = glob(f"{class_name}_batch_*")
    total = sum(len(glob(os.path.join(b, "*.wav"))) for b in batches)
    print(f"{class_name}: {total} clips across {len(batches)} batches")


🔧 FFmpeg path detected by Pydub: C:\Users\gaurav\Desktop\ffmpeg.exe
🔍 [man-woman-ailing-screaming-in-pain] Searching YouTube for: man screaming in pain sound effect
⬇ [man-woman-ailing-screaming-in-pain] Downloading from: Man Screaming in Pain Sound Effect
✅ [man-woman-ailing-screaming-in-pain] Saved: man-woman-ailing-screaming-in-pain_audio_file1678.wav (1678/2000)
⬇ [man-woman-ailing-screaming-in-pain] Downloading from: Man Screaming Sound Effect (READ DESC for MP3/FORM)
⬇ [man-woman-ailing-screaming-in-pain] Downloading from: Free Sound effect of male man screaming in pain and fear
✅ [man-woman-ailing-screaming-in-pain] Saved: man-woman-ailing-screaming-in-pain_audio_file1679.wav (1679/2000)
✅ [man-woman-ailing-screaming-in-pain] Saved: man-woman-ailing-screaming-in-pain_audio_file1680.wav (1680/2000)
⬇ [man-woman-ailing-screaming-in-pain] Downloading from: Man Upset / Raging Screams Sound Effect
✅ [man-woman-ailing-screaming-in-pain] Saved: man-woman-ailing-screaming-in-pain_audio_